In [4]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import sys
# Add the utils folder to the system path so we can import custom modules
sys.path.append(os.path.abspath(os.path.join("../..")))
from utils.reproducibility_utils import seed_everything
from utils.data_loadingbins import load_data, load_test_data

# --- 1. SETTINGS & PATHS ---
TRAIN_PATH = '../../datasets_challenge/train'
DIST_CSV = '../../datasets_challenge/sideSlip_distribution.csv'
MODEL_DIR = './expert_models'
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42) # Use your existing reproducibility util

# --- 2. MODEL DEFINITION ---
class RNN(nn.Module):
    def __init__(self, input_size, output_size, rnn_units, fc_units, dropout=[0.0, 0.0, 0.0]):
        super(RNN, self).__init__()
        
        self.input_dropout = nn.Dropout(dropout[0])
        self.rnn = nn.GRU(input_size, rnn_units, batch_first=True)
        self.rnn_dropout = nn.Dropout(dropout[1])
        self.fc = nn.Linear(rnn_units, fc_units)
        self.fc_dropout = nn.Dropout(dropout[2])
        self.output = nn.Linear(fc_units, output_size)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.input_dropout(x)
        rnn_out, _ = self.rnn(x)
        rnn_out = self.rnn_dropout(rnn_out[:, -1, :])  # Take the last time step
        fc_out = self.relu(self.fc(rnn_out))
        fc_out = self.fc_dropout(fc_out)
        out = self.output(fc_out)
        return out


# --- 3. WEIGHTED LOSS (Targeting Few-Shot) ---
def weighted_huber_loss(predictions, targets, delta=1.0):
    loss = torch.nn.functional.huber_loss(predictions.squeeze(), targets, delta=delta, reduction='none')
    weights = torch.ones_like(targets)
    weights[targets.abs() > 2.3] = 15.0  # Medium
    weights[targets.abs() > 5.0] = 50.0  # Few
    return (weights * loss).mean()

# --- 4. TRAINING FUNCTION ---
def train_expert(model, loader, optimizer, epochs=15):
    model.train()
    for epoch in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            preds = model(x)
            loss = weighted_huber_loss(preds, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

# --- 5. MAIN EXECUTION: TRAIN ALL EXPERTS ---
all_files = sorted([os.path.basename(f) for f in glob.glob(os.path.join(TRAIN_PATH, '*.csv'))])
# Optional: Only use files with significant data to save time
# all_files = ['train_part_017.csv', 'train_part_005.csv', ...] 

model_paths = []
# Initialize and Train
device = "cpu"  # Change to "cuda" if you have a GPU available

# Model hyperparameters
input_size = 22  # Number of features (22 in this case)
rnn_units = 48  # Number of units in RNN hidden state
fc_units = 48  # Number of units in fully connected layer
output_size = 1  # Single output (sideslip angle)
lr = 0.0003  # Learning rate for optimizer

print(f"Starting Training for {len(all_files)} Experts...")

for val_file in all_files:
    save_path = os.path.join(MODEL_DIR, f"expert_{val_file}.pth")
    
    try:
        # Load data for this specific split
        train_loader, _, pipeline = load_data(
            train_path=TRAIN_PATH,
            dist_csv_path=DIST_CSV,
            val_file_name=val_file, 
            columns_to_drop=["sideSlip", "vyCG", "time", "longitude","latitude"],
            sequence_length=20,
            threshold=True,
            threshold_value=3.0,
            threshold_column="vxCG",
            target='sideSlip',
            batchsize=64
        )
        
        # Initialize the model instance properly
        model = RNN(
            input_size=input_size, 
            output_size=output_size,
            rnn_units=rnn_units,
            fc_units=fc_units
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        
        print(f"Training Expert: {val_file}")
        train_expert(model, train_loader, optimizer, epochs=10)
        
        torch.save(model.state_dict(), save_path)
        model_paths.append(save_path)
        
    except ValueError as e:
        # This catches the "No objects to concatenate" if a file results in 0 samples
        print(f"Skipping {val_file}: {e}")
        continue

# --- 6. ENSEMBLE PREDICTION ON TEST DATA ---
def ensemble_predict(test_loader, paths):
    all_expert_preds = []
    
    # Use the same model architecture
    # FIX: Initialize an instance of the model with parameters
    model = RNN(
        input_size=input_size, 
        output_size=output_size,
        rnn_units=rnn_units,
        fc_units=fc_units
    ).to(device)
    
    print("\nRunning Ensemble Inference...")
    for path in paths:
        print(f"Loading {os.path.basename(path)}...")
        model.load_state_dict(torch.load(path, map_location=device))
        model.eval()
        
        preds = []
        with torch.no_grad():
            for x, _ in test_loader: # Assuming test_loader yields (data, dummy_targets)
                out = model(x.to(device))
                preds.append(out.cpu().numpy())
        
        all_expert_preds.append(np.concatenate(preds))
    
    # Stack and calculate the average across all experts
    # Shape: (num_experts, num_test_samples)
    stacked_preds = np.stack(all_expert_preds)
    final_avg = np.mean(stacked_preds, axis=0)
    
    return final_avg


Starting Training for 22 Experts...
VALIDATION: Loading train_part_001.csv
Calculating weights for WeightedRandomSampler...
Training Expert: train_part_001.csv
VALIDATION: Loading train_part_002.csv
Calculating weights for WeightedRandomSampler...
Skipping train_part_002.csv: No objects to concatenate
VALIDATION: Loading train_part_003.csv
Calculating weights for WeightedRandomSampler...
Training Expert: train_part_003.csv
VALIDATION: Loading train_part_004.csv
Calculating weights for WeightedRandomSampler...
Training Expert: train_part_004.csv
VALIDATION: Loading train_part_005.csv
Calculating weights for WeightedRandomSampler...
Training Expert: train_part_005.csv
VALIDATION: Loading train_part_006.csv
Calculating weights for WeightedRandomSampler...
Training Expert: train_part_006.csv
VALIDATION: Loading train_part_007.csv
Calculating weights for WeightedRandomSampler...
Training Expert: train_part_007.csv
VALIDATION: Loading train_part_008.csv
Calculating weights for WeightedRandom

In [5]:
print("\nLoading test data...")
test_dataloader = load_test_data(test_path = "../../datasets_challenge/test",
                                 batchsize = 64,
                                 pipeline = pipeline, 
                                 load_anonymized_data=True)


Loading test data...
../../datasets_challenge/test\test_part_001.csv
../../datasets_challenge/test\test_part_002.csv
../../datasets_challenge/test\test_part_003.csv
../../datasets_challenge/test\test_part_004.csv
../../datasets_challenge/test\test_part_005.csv
../../datasets_challenge/test\test_part_006.csv
../../datasets_challenge/test\test_part_007.csv
../../datasets_challenge/test\test_part_008.csv
../../datasets_challenge/test\test_part_009.csv
../../datasets_challenge/test\test_part_010.csv
../../datasets_challenge/test\test_part_011.csv


In [6]:

final_predictions = ensemble_predict(test_dataloader, model_paths)
print("Ensemble prediction complete.")


Running Ensemble Inference...
Loading expert_train_part_001.csv.pth...
Loading expert_train_part_003.csv.pth...
Loading expert_train_part_004.csv.pth...
Loading expert_train_part_005.csv.pth...
Loading expert_train_part_006.csv.pth...
Loading expert_train_part_007.csv.pth...
Loading expert_train_part_008.csv.pth...
Loading expert_train_part_009.csv.pth...
Loading expert_train_part_010.csv.pth...
Loading expert_train_part_011.csv.pth...
Loading expert_train_part_012.csv.pth...
Loading expert_train_part_013.csv.pth...
Loading expert_train_part_015.csv.pth...
Loading expert_train_part_017.csv.pth...
Loading expert_train_part_018.csv.pth...
Loading expert_train_part_019.csv.pth...
Loading expert_train_part_020.csv.pth...
Loading expert_train_part_021.csv.pth...
Loading expert_train_part_022.csv.pth...
Ensemble prediction complete.


In [8]:
# Flatten the predictions to ensure they are 1D
flat_predictions = np.array(final_predictions).flatten()

# Save predictions in the required format for submission
y_pred = pd.DataFrame({
    "id": range(len(flat_predictions)),
    "predicted_sideslip": flat_predictions
})

# Reorder columns just in case the competition expects 'id' first
y_pred = y_pred[["id", "predicted_sideslip"]]

# Save to CSV file
y_pred.to_csv("predictionsemsembleval.csv", index=False)

print(f"✓ Predictions saved to 'predictionsemsembleval.csv'")
print(f"✓ Total samples: {len(flat_predictions)}")

✓ Predictions saved to 'predictionsemsembleval.csv'
✓ Total samples: 49328
